# 🏥 AI Clinical Decision Support Lite - Day 1 & 2
This notebook demonstrates the execution of the research-informed clinical RAG implementation.


## Setup & Dependencies


In [ ]:
# Import the `userdata` module to securely access your GitHub Token
from google.colab import userdata
import os

# Get the GH_TOKEN from Colab Secrets (if available)
# userdata.get() returns None if the key is not found
try:
    GH_TOKEN = userdata.get('GH_TOKEN')
except:
    GH_TOKEN = None

# Construct the repository URL, including the token if it exists
repo_url = "https://github.com/Abdelrahmann-Mostafa/Pyramind---Hackathon.git"
if GH_TOKEN is not None:
    # Insert the token into the URL for authentication
    repo_url = repo_url.replace("https://", f"https://oauth2:{GH_TOKEN}@")

# Clone the repository
!git clone {repo_url}

print("Repository cloned successfully!")


In [ ]:
# Move the 'data' and 'src' directories from the cloned repository to the current working directory

# Define the path to the cloned repository
cloned_repo_path = "./Pyramind---Hackathon"

# Move 'data' directory
if os.path.exists(os.path.join(cloned_repo_path, "data")):
    !mv {cloned_repo_path}/data .
    print("Moved 'data' directory.")
else:
    print("'data' directory not found in the cloned repository.")

# Move 'src' directory
if os.path.exists(os.path.join(cloned_repo_path, "src")):
    !mv {cloned_repo_path}/src .
    print("Moved 'src' directory.")
else:
    print("'src' directory not found in the cloned repository.")


In [ ]:
!pip install sentence-transformers chromadb pydantic torch PyMuPDF
import os
import sys
from pathlib import Path

# Ensure src is in python path
PROJECT_ROOT = str(Path(os.getcwd()).parent)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Or if notebook is in root
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())


## Day 1: Document Ingestion & Embedding Generation


In [ ]:
from src.ingestion import run_ingestion_pipeline
from src.embed_chunks import run_embedding_pipeline

# 1. Run Ingestion (Extracts PDFs, Hierarchical Chunking)
chunks = run_ingestion_pipeline()

# 2. Run Embedding (Generates Dense Embeddings for Child Chunks)
embedded_chunks = run_embedding_pipeline()


## Day 2: Retrieval Optimization & Evaluation


### 1. Import Retrieval Module


In [ ]:
from src.retrieval_optimization import create_chroma_index, load_embedding_model, retrieve_with_config, compress_context, CHROMA_PATH, EMBEDDINGS_FILE, MODEL_NAME
from src.retrieval_optimization import evaluate_retrieval, test_hyperparameters, run_evaluation_pipeline


### 2. Create ChromaDB Index


In [ ]:
collection = create_chroma_index(embeddings_file=EMBEDDINGS_FILE, chroma_path=CHROMA_PATH)
embedding_model = load_embedding_model(MODEL_NAME)
print(f"Collection created with {collection.count()} chunks.")


### 3. Load Benchmark Queries


In [ ]:
import json
with open('data/benchmark_queries.json', 'r', encoding='utf-8') as f:
    queries = json.load(f)
print(f"Loaded {len(queries)} benchmark queries.")
print("Sample query:", queries[0]['query'])


### 4. Test Optimal Configuration (k=3, with compression)


In [ ]:
query_text = queries[0]['query']
results = retrieve_with_config(query_text, collection, embedding_model, k=3, compress=True)

print(f"Query: {query_text}")
for res in results:
    print(f"\nRank {res['rank']} (Score: {res['similarity_score']:.4f})")
    print(f"Doc: {res['document_name']} | Sec: {res['section_title']} | Pop: {res['target_population']}")
    print(f"Content: {res['content'][:150]}...")


### 5. Run Full Hyperparameter Evaluation


In [ ]:
log = run_evaluation_pipeline()
